# Simple Python Dynamics Model

> This script generates an estimated model for Icarus utilizing a Panda Data Frame to store all variables.

## Data Frame Properties
This data frame is formated that each row is a the rocket's current state at a specific time.

## Model Variables
List all variables this model calculates for
1. **Time** → Measured in seconds ($s$)
2. **Altitude** → Measured in feet ($m$)
3. **Velocity** → Measured in feet per second ($\frac{m}{s}$)
4. **Acceleration** → Measured in g's ($n = \frac{a}{g}$)
5. **Mass** → Measured in pound mass ($kg$)
6. **Mass Flow Rate** → Measured in pound mass per second ($\frac{kg}{s}$)
7. **Thrust** → Measured in pound force ($N$)
8. **Surface Area** → Measured in feet squared ($m^2$)
9. **Coefficient of Drag** → Unitless
10. **isFirstParachuteOpen** → Unitless
11. **isSecondParachuteOpen** → Unitless
12. **isEngineOne** → Unitless

## Contributors
**Tanner Ross, Abigail Dickson, Zeshan Javaid**



In [39]:
# Imports
import pandas as pd
import numpy as np
import csv
from google.colab import files

uploaded_csv = files.upload() # UPLOAD BOTH FILES: atm.csv and thrust.csv in this exact order

Saving atm.csv to atm (2).csv
Saving thrust.csv to thrust (2).csv


#### Compute Acceleration
**The following function computes the acceleration**
##### Implementation
Implements the following formula

$$
        \sum{F} = T - D - W = ma + \sum{\dot{m}}v \\
        T - \frac{1}{2} \rho v^2 S C_d - mg = ma + (\dot{m_{fuel}}+ \dot{m_{N_2O}})v \\
        \boxed{ n = \frac{a}{g} = \frac{ T - \frac{1}{2} \rho v^2 S C_d - (\dot{m_{fuel}}+ \dot{m_{N_2O}})v }{mg} - 1 }
$$

##### Parameters
- Thrust → Measured in $N$
- Altitude → Measured in $m$
- Velocity → Measured in $m/s$
- Surface Area → Measured in $m^2$
- Coefficient of Drag → Unitless
- Mass → Measured in $kg$
- Mass Flow Rate → Measured in $\frac{kg}{s}$

##### Returns
- Returns acceleration in terms of g's, $n$ where $n=\frac{a}{g}$.



In [40]:
def get_density(altitude):
  with open(list(uploaded_csv.keys())[0],"r") as file:
    rows = file.readlines()
    for index,row in enumerate(rows):
      row = [float(r) for r in row.split(",")]
      if(altitude < row[0] and index > 0):
        prev_row = [float(r) for r in rows[index - 1].split(",")]
        slope = ((row[2] - prev_row[2]) / (row[0] - prev_row[0])) # slope (p2 - p1) / (h2 - h1)
        return prev_row[2] + slope * (altitude - prev_row[0]) # Using linear interpolation: p = p_o + slope(h - h_0)
      elif(index == 0):
        return row[2]
      else:
        print("ERROR: INVALID ALTITUDE")
        return -1

def compute_acceleration(thrust, altitude, velocity, surface_area, coefficient_of_drag, mass, mass_flow_rate):
  drag = 1 / 2 * get_density(altitude) * velocity ** 2 * surface_area * coefficient_of_drag
  mass_change_force_component = (mass_flow_rate) * velocity
  return (thrust - drag - mass_change_force_component) / (mass * 9.81) - 1



#### Compute Velocity
**The following function computes the velocity**

##### Implementation
Implements the following formula: $$v = v_o + a(t-t_0) \\ a = ng \\ v = v_o + ng(t-t_0)$$

##### Parameters
- Acceleration → Measured in $\frac{m}{s}$
- Initial Velocity → Measured in $\frac{m}{s}$
- Initial Time → Measured in $s$
- Current Time → Measured in $s$

##### Returns
- Velocity → Measured in $\frac{m}{s}$

In [41]:
def compute_velocity(acceleration, prev_time, curr_time, prev_velocity):
  return prev_velocity + (acceleration * 9.81) * (curr_time - prev_time)

#### Compute Altitude
**The following function computes the velocity**

##### Implementation
The following function implements the following formula:
$$
y = \int_{t_0}^{t}{v(t)dt}=\int_{t_0}^{t}{(v_0 + at)dt} = y_o + v_ot + \frac{1}{2}at^2 \\ a = ng \\ y_o + v_ot + \frac{1}{2}ngt^2
$$

##### Parameters
- Acceleration → Measured in $\frac{m}{s}$
- Initial Velocity → Measured in $\frac{m}{s}$
- Initial Altitude → Measured in $m$
- Initial Time → Measured in $s$
- Current Time → Measured in $s$

##### Returns
- Current Altitude → Measured in $m$

In [42]:
def compute_altitude(acceleration, prev_time, curr_time, prev_velocity, prev_altitude):
  return prev_altitude + prev_velocity * (curr_time - prev_time) * 1 / 2 + (acceleration * 9.81) * (curr_time - prev_time) ** 2

#### Compute ____
**The following function computes the ___**

##### Implementation

##### Parameters

##### Returns

# Main Function
**Connects all the functions together**

In [54]:

def main():
  with open(list(uploaded_csv.keys())[1],"r", encoding='utf-8') as file:
    # Define the dataframe with headers

    SimulationDataFrame = pd.DataFrame(columns=["Time (s)",
                                                "Altitude (m)",
                                                "Velocity (m/s)",
                                                "Acceleration (g\'s)",
                                                "Mass (kg)",
                                                "Mass Flow (kg / s)",
                                                "Thrust (N)",
                                                "Surface Area (m^2)",
                                                "Coefficient of Drag (Unitless)",
                                                "isFirstParachuteOpen (Unitless)",
                                                "isSecondParachuteOpen (Unitless)",
                                                "isEngineOn (Unitless)"])

    thrust_profile = csv.DictReader(file) # Reads csv into a list of dictionary where keys are the headers

    # Start with initial case
    prev_time, time, prev_altitude, altitude, prev_velocity, velocity, acceleration, mass, mass_flow, surface_area, Cd, isFirstParachuteOpen, isSecondParachuteOpen, isEngineOn = 0, 0, 0, 0, 0, 0, -1, 25, -1, -1, 0, False, False, True  # TODO: change once found
    aerodynamicProperties = {
        "withoutParachute": {"Cd" : 0.1,
                          "S" : 4},
        "withFirstParachute": {"Cd" : 0.5,
                            "S" : 8},
        "withSecondParachute": {"Cd" : 0.9,
                            "S" : 16}
    } # TODO: Implement RasAERO to find actual values for Cd
    for row in thrust_profile:

      # Kinematic Logic
      time = float(row["\ufeffTime"])
      mass -= mass_flow * (time - prev_time)
      acceleration = compute_acceleration(float(row["F"]), prev_altitude, prev_velocity, surface_area, Cd, mass, float(row["m_dot"]))
      velocity = compute_velocity(acceleration, prev_time, time, prev_velocity)
      altitude = compute_altitude(acceleration, prev_time, time, prev_velocity, prev_altitude)
      Cd = aerodynamicProperties["withoutParachute"]["Cd"]
      surface_area = aerodynamicProperties["withoutParachute"]["S"]

      # Engine On Logic
      isEngineOn = row["Status"]=="Burning"

      # Add data into main data frame
      SimulationDataFrame.loc[len(SimulationDataFrame)] = [time, altitude, velocity, acceleration, mass, float(row["m_dot"]), float(row["F"]), surface_area, Cd, isFirstParachuteOpen, isSecondParachuteOpen, isEngineOn]

      # assign previous variables
      prev_time = time
      prev_altitude = altitude
      prev_velocity = velocity

    while altitude > 0:

      # Kinematic Logic
      time += 1
      acceleration = compute_acceleration(0, prev_altitude, prev_velocity, surface_area, Cd, mass, 0)
      velocity = compute_velocity(acceleration, prev_time, time, prev_velocity)
      altitude = compute_altitude(acceleration, prev_time, time, prev_velocity, prev_altitude)

      # Parachute logic
      if(velocity <= 0):
        if(altitude < 300): # TODO: Set actual value here for second parachute deployment
          isFirstParachuteOpen = True
          isSecondParachuteOpen = True
          Cd = aerodynamicProperties["withSecondParachute"]["Cd"]
          surface_area = aerodynamicProperties["withSecondParachute"]["S"]
        else:
          isFirstParachuteOpen = True
          if(isFirstParachuteOpen):
            Cd = aerodynamicProperties["withFirstParachute"]["Cd"]
            surface_area = aerodynamicProperties["withFirstParachute"]["S"]
      else:
        Cd = aerodynamicProperties["withoutParachute"]["Cd"]
        surface_area = aerodynamicProperties["withoutParachute"]["S"]

      # Add data to dataframe
      SimulationDataFrame.loc[len(SimulationDataFrame)] = [time, altitude, velocity, acceleration, mass, 0, 0, surface_area, Cd, isFirstParachuteOpen, isSecondParachuteOpen, isEngineOn]

      # assign previous variables
      prev_time = time
      prev_altitude = altitude
      prev_velocity = velocity



    print(print(SimulationDataFrame.loc[25:30]))


if __name__ == "__main__":
  main()

    Time (s)  Altitude (m)  Velocity (m/s)  Acceleration (g's)  Mass (kg)  \
25     1.015     33.548044       90.264425            4.484059     26.015   
26     1.115     38.418573       93.837502            3.642281     26.115   
27     1.215     43.396438       96.697409            2.915298     26.215   
28     1.315     48.457368       98.957996            2.304370     26.315   
29     1.415     53.583286      100.738186            1.814669     26.415   
30     1.515     58.760213      102.138358            1.427291     26.515   

    Mass Flow (kg / s)  Thrust (N)  Surface Area (m^2)  \
25               2.461      3495.0                   4   
26               2.455      3493.0                   4   
27               2.450      3487.0                   4   
28               2.446      3479.0                   4   
29               2.447      3474.0                   4   
30               2.445      3471.0                   4   

    Coefficient of Drag (Unitless)  isFirstParachuteO

In [ ]:
main()